## Step 1A — Parse TSV into clean closed boundary vertex lists

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Parse `parks.tsv` and extract one ordered boundary vertex list per valid shape.

### Supported geometry types
- `LINESTRING` -> use coordinates only if closed
- `POLYGON` -> use exterior ring
- `MULTIPOLYGON` -> use largest polygon exterior
- `GEOMETRYCOLLECTION` -> use largest polygon if polygon exists, otherwise closed linestring if present

### Current baseline rule
- Keep only closed boundaries
- Remove duplicated closing point
- Skip invalid/open geometries

### Output
`polygons = [(poly_id, vertices, metadata), ...]`

In [17]:
# ============================================================
# Step 1A — Parse TSV into clean closed boundary vertex lists
# Type: Pipeline Code
# Keep later? Yes
# ============================================================

import csv
from shapely import wkt
from shapely.geometry import Polygon, MultiPolygon, GeometryCollection, LineString
from tqdm import tqdm

DATA_PATH = "/raid/ruban/data/parks.tsv"


def is_closed_linestring(ls):
    coords = list(ls.coords)
    return len(coords) >= 4 and coords[0] == coords[-1]


def remove_duplicate_closing_point(coords):
    coords = list(coords)
    if len(coords) >= 2 and coords[0] == coords[-1]:
        coords = coords[:-1]
    return coords


def geometry_to_vertex_list(geom):
    """
    Convert supported geometry -> ordered boundary vertex list.

    Rules:
      - Polygon -> exterior ring
      - MultiPolygon -> largest polygon by area
      - GeometryCollection -> prefer largest polygon; else closed LineString
      - LineString -> only if closed
    """
    if isinstance(geom, Polygon):
        coords = remove_duplicate_closing_point(geom.exterior.coords)
        if len(coords) < 3:
            raise ValueError("Polygon has fewer than 3 unique vertices")
        return [(float(x), float(y)) for x, y in coords]

    if isinstance(geom, MultiPolygon):
        polys = list(geom.geoms)
        if not polys:
            raise ValueError("Empty MultiPolygon")
        main_poly = max(polys, key=lambda p: p.area)
        coords = remove_duplicate_closing_point(main_poly.exterior.coords)
        if len(coords) < 3:
            raise ValueError("MultiPolygon exterior has fewer than 3 unique vertices")
        return [(float(x), float(y)) for x, y in coords]

    if isinstance(geom, LineString):
        if not is_closed_linestring(geom):
            raise ValueError("Open LineString")
        coords = remove_duplicate_closing_point(geom.coords)
        if len(coords) < 3:
            raise ValueError("Closed LineString has fewer than 3 unique vertices")
        return [(float(x), float(y)) for x, y in coords]

    if isinstance(geom, GeometryCollection):
        polys = [g for g in geom.geoms if isinstance(g, Polygon)]
        if polys:
            main_poly = max(polys, key=lambda p: p.area)
            coords = remove_duplicate_closing_point(main_poly.exterior.coords)
            if len(coords) < 3:
                raise ValueError("GeometryCollection polygon has fewer than 3 unique vertices")
            return [(float(x), float(y)) for x, y in coords]

        closed_lines = [g for g in geom.geoms if isinstance(g, LineString) and is_closed_linestring(g)]
        if closed_lines:
            main_line = max(closed_lines, key=lambda g: len(list(g.coords)))
            coords = remove_duplicate_closing_point(main_line.coords)
            if len(coords) < 3:
                raise ValueError("GeometryCollection closed LineString has fewer than 3 unique vertices")
            return [(float(x), float(y)) for x, y in coords]

        raise ValueError("GeometryCollection has no usable polygon or closed linestring")

    raise ValueError(f"Unsupported geometry: {geom.geom_type}")


def parse_parks_tsv_row(row):
    if len(row) < 2:
        raise ValueError("Row has fewer than 2 columns")

    poly_id = row[0].strip()
    wkt_text = row[1].strip()
    metadata = row[2].strip() if len(row) > 2 else None

    geom = wkt.loads(wkt_text)
    vertices = geometry_to_vertex_list(geom)

    return poly_id, vertices, metadata


# ======== MAIN LOADING LOOP ========

polygons = []
skipped = 0
skip_reasons = {}

with open(DATA_PATH, "r", encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")

    for row in tqdm(reader, desc="Parsing closed boundaries"):
        try:
            poly_id, vertices, metadata = parse_parks_tsv_row(row)
            polygons.append((poly_id, vertices, metadata))
        except Exception as e:
            skipped += 1
            reason = str(e)
            skip_reasons[reason] = skip_reasons.get(reason, 0) + 1

print("\n=== Parsing Summary ===")
print(f"Total valid boundaries loaded : {len(polygons)}")
print(f"Skipped rows                  : {skipped}")

print("\nTop skip reasons:")
for reason, count in sorted(skip_reasons.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"{reason}: {count}")

Parsing closed boundaries: 234447it [00:49, 4706.26it/s] 


=== Parsing Summary ===
Total valid boundaries loaded : 233551
Skipped rows                  : 896

Top skip reasons:
Open LineString: 896


## M2 Upgrade Step 1 — Build richer boundary-geometry node features

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Upgrade each polygon graph so every node contains local boundary-geometry features, not just raw coordinates.

### New node features
For each vertex:
- `x, y`
- previous edge vector: `dx_prev, dy_prev`
- next edge vector: `dx_next, dy_next`
- previous edge length: `len_prev`
- next edge length: `len_next`
- turning-angle features: `cos_turn, sin_turn`

### Output
Each graph now has node feature size:
- `10`

In [29]:
# ============================================================
# M2 Upgrade Step 1 — Build richer boundary-geometry node features
# Type: Pipeline Code
# Keep later? Yes
# Replace old Step 1B code cell with this version
# ============================================================

import torch
from torch_geometric.data import Data


def build_cycle_edge_index(num_nodes):
    """
    Build an undirected cycle graph for a polygon boundary.
    """
    if num_nodes < 3:
        raise ValueError("A valid polygon graph needs at least 3 nodes")

    src = []
    dst = []

    for i in range(num_nodes):
        j = (i + 1) % num_nodes

        src.append(i)
        dst.append(j)

        src.append(j)
        dst.append(i)

    edge_index = torch.tensor([src, dst], dtype=torch.long)
    return edge_index


def build_boundary_geometry_features(vertices, eps=1e-12):
    """
    Build richer node features for a polygon boundary.

    Output feature layout:
      0: x
      1: y
      2: dx_prev
      3: dy_prev
      4: dx_next
      5: dy_next
      6: len_prev
      7: len_next
      8: cos_turn
      9: sin_turn
    """
    xy = torch.tensor(vertices, dtype=torch.float32)   # [N, 2]

    xy_prev = torch.roll(xy, shifts=1, dims=0)
    xy_next = torch.roll(xy, shifts=-1, dims=0)

    vec_prev = xy - xy_prev
    vec_next = xy_next - xy

    dx_prev = vec_prev[:, 0:1]
    dy_prev = vec_prev[:, 1:2]
    dx_next = vec_next[:, 0:1]
    dy_next = vec_next[:, 1:2]

    len_prev = torch.norm(vec_prev, dim=1, keepdim=True)
    len_next = torch.norm(vec_next, dim=1, keepdim=True)

    dir_prev = vec_prev / (len_prev + eps)
    dir_next = vec_next / (len_next + eps)

    cos_turn = torch.sum(dir_prev * dir_next, dim=1, keepdim=True)

    sin_turn = (
        dir_prev[:, 0:1] * dir_next[:, 1:2]
        - dir_prev[:, 1:2] * dir_next[:, 0:1]
    )

    x_feat = torch.cat([
        xy,
        dx_prev,
        dy_prev,
        dx_next,
        dy_next,
        len_prev,
        len_next,
        cos_turn,
        sin_turn,
    ], dim=1)

    return x_feat


def polygon_to_pyg_data(poly_id, vertices, metadata=None):
    """
    Convert one polygon boundary into a PyG Data object
    with richer local boundary-geometry node features.
    """
    if len(vertices) < 3:
        raise ValueError("Polygon must have at least 3 vertices")

    x = build_boundary_geometry_features(vertices)   # [num_nodes, 10]
    edge_index = build_cycle_edge_index(len(vertices))

    data = Data(x=x, edge_index=edge_index)
    data.poly_id = str(poly_id)
    data.metadata = metadata if metadata is not None else ""

    return data

## Step 1C — Convert full dataset into PyTorch Geometric graphs

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Convert all parsed polygon boundaries into PyTorch Geometric `Data` objects.

### Input
`polygons = [(poly_id, vertices, metadata), ...]`

### Output
`graph_data_list = [Data(...), Data(...), ...]`

### Tracking
This step also records:
- number of successfully built graphs
- skipped graphs
- node count statistics

### Notes
This is part of the actual PolygonGNN pipeline.

In [30]:
# ============================================================
# Step 1C — Convert full dataset into PyG graphs
# Type: Pipeline Code
# Keep later? Yes
# ============================================================

from tqdm import tqdm

graph_data_list = []
graph_skipped = 0
graph_skip_reasons = {}

node_counts = []

for poly_id, vertices, metadata in tqdm(polygons, desc="Building PyG graphs"):
    try:
        g = polygon_to_pyg_data(poly_id, vertices, metadata)
        graph_data_list.append(g)
        node_counts.append(g.num_nodes)
    except Exception as e:
        graph_skipped += 1
        reason = str(e)
        graph_skip_reasons[reason] = graph_skip_reasons.get(reason, 0) + 1

print("\n=== Graph Build Summary ===")
print(f"Graphs built successfully : {len(graph_data_list)}")
print(f"Graphs skipped            : {graph_skipped}")

if node_counts:
    print(f"Min nodes per graph       : {min(node_counts)}")
    print(f"Max nodes per graph       : {max(node_counts)}")
    print(f"Avg nodes per graph       : {sum(node_counts) / len(node_counts):.2f}")

if graph_skip_reasons:
    print("\nTop graph skip reasons:")
    for reason, count in sorted(graph_skip_reasons.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"{reason}: {count}")

Building PyG graphs: 100%|██████████| 233551/233551 [01:44<00:00, 2242.40it/s]


=== Graph Build Summary ===
Graphs built successfully : 233551
Graphs skipped            : 0
Min nodes per graph       : 3
Max nodes per graph       : 1894
Avg nodes per graph       : 15.55


## Step 1D — Normalize polygon graph coordinates

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Normalize each graph's node coordinates so the model learns shape rather than absolute location or size.

### Normalization
For each graph:
1. Compute centroid of vertex coordinates
2. Subtract centroid from all vertices
3. Divide by maximum distance from centroid

### Result
Each polygon becomes:
- centered at origin
- scaled to comparable size

### Notes
This is part of the actual PolygonGNN pipeline.

In [31]:
# ============================================================
# M2 Upgrade Step 2 — Normalize richer 10D boundary-geometry node features
# Type: Pipeline Code
# Keep later? Yes
# Replace old Step 1D normalization function with this version
# ============================================================

import torch

def normalize_graph_coordinates(data, eps=1e-12):
    """
    Normalize upgraded 10D boundary-geometry node features.

    Expected feature layout:
      0: x
      1: y
      2: dx_prev
      3: dy_prev
      4: dx_next
      5: dy_next
      6: len_prev
      7: len_next
      8: cos_turn
      9: sin_turn
    """
    x = data.x.clone()

    if x.ndim != 2 or x.shape[1] != 10:
        raise ValueError(f"Expected x to have shape [num_nodes, 10], got {tuple(x.shape)}")

    # ---- coordinates ----
    xy = x[:, 0:2]                                # [N, 2]
    center = xy.mean(dim=0, keepdim=True)         # [1, 2]
    xy_centered = xy - center

    radii = torch.norm(xy_centered, dim=1)        # [N]
    max_radius = radii.max()

    if max_radius < eps:
        scale = torch.tensor(1.0, dtype=x.dtype, device=x.device)
    else:
        scale = max_radius

    # normalize coordinates
    x[:, 0:2] = xy_centered / scale

    # normalize edge vectors with same scale
    x[:, 2:6] = x[:, 2:6] / scale

    # normalize edge lengths
    x[:, 6:8] = x[:, 6:8] / scale

    # keep turn-angle features unchanged
    # x[:, 8:10] stays the same

    data.x = x
    return data

In [32]:
# ============================================================
# Step 1D — Normalize all graphs
# Type: Pipeline Code
# Keep later? Yes
# ============================================================

from tqdm import tqdm

normalized_graph_data_list = []

for g in tqdm(graph_data_list, desc="Normalizing graphs (10D features)"):
    g_norm = normalize_graph_coordinates(g)
    normalized_graph_data_list.append(g_norm)

print("\n=== Normalization Summary ===")
print(f"Normalized graphs: {len(normalized_graph_data_list)}")

Normalizing graphs (10D features): 100%|██████████| 233551/233551 [00:41<00:00, 5565.26it/s]


=== Normalization Summary ===
Normalized graphs: 233551


## REAL GT PIPELINE STARTS HERE

**Type:** Final evaluation/training alignment  
**Keep later?** Yes

### Purpose
From this point onward, the notebook switches from the earlier random-split / pseudo-evaluation path to the real ground-truth-aligned pipeline.

### Ground-truth facts
- GT uses global dataset row indices
- database polygons = indices `< 187019`
- query polygons = indices `>= 187019`
- GT exists only for the query set

### Important
All final retrieval evaluation must use this GT-aligned split, not the earlier random split.

In [33]:
# ============================================================
# GT Step 1 — Attach global dataset indices and build DB / Query split
# Type: Pipeline Code
# Keep later? Yes
# ============================================================

QUERY_START_IDX = 187019

for global_idx, g in enumerate(normalized_graph_data_list):
    g.global_idx = global_idx

graphs_by_global_idx = {g.global_idx: g for g in normalized_graph_data_list}

db_graphs_gt = [g for g in normalized_graph_data_list if g.global_idx < QUERY_START_IDX]
query_graphs_gt = [g for g in normalized_graph_data_list if g.global_idx >= QUERY_START_IDX]

print("=== GT-Aligned Split Summary ===")
print(f"Total graphs      : {len(normalized_graph_data_list)}")
print(f"DB graphs         : {len(db_graphs_gt)}")
print(f"Query graphs      : {len(query_graphs_gt)}")
print(f"Expected query start idx: {QUERY_START_IDX}")
print(f"First query global_idx  : {query_graphs_gt[0].global_idx if query_graphs_gt else 'N/A'}")
print(f"Last DB global_idx      : {db_graphs_gt[-1].global_idx if db_graphs_gt else 'N/A'}")

=== GT-Aligned Split Summary ===
Total graphs      : 233551
DB graphs         : 187019
Query graphs      : 46532
Expected query start idx: 187019
First query global_idx  : 187019
Last DB global_idx      : 187018


In [34]:
# ============================================================
# GT Step 1 Test — Inspect DB / Query alignment
# Type: Diagnostic / Inspection Code
# Keep later? No
# ============================================================

print("DB sample:")
print("global_idx:", db_graphs_gt[0].global_idx, "| poly_id:", db_graphs_gt[0].poly_id, "| nodes:", db_graphs_gt[0].num_nodes)

print("\nQuery sample:")
print("global_idx:", query_graphs_gt[0].global_idx, "| poly_id:", query_graphs_gt[0].poly_id, "| nodes:", query_graphs_gt[0].num_nodes)

DB sample:
global_idx: 0 | poly_id: 4061698 | nodes: 56

Query sample:
global_idx: 187019 | poly_id: 34775349 | nodes: 13


## GT Step 2 — Load real similarity map from ground-truth files

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Parse the real similarity-map files into a usable ground-truth dictionary.

### Format
Each line looks like:
`query_idx, neighbor1, neighbor2, neighbor3, ...`

### Output
- `gt_map`
  where:
  `gt_map[query_idx] = [neighbor_idx1, neighbor_idx2, ...]`

In [35]:
# ============================================================
# GT Step 2 — Load real similarity map from ground-truth files
# Type: Pipeline Code
# Keep later? Yes
# ============================================================

import os
from tqdm import tqdm

GT_DIR = "/raid/ruban/groundtruth/pk-query-187019"


def load_similarity_map(gt_dir):
    gt_files = sorted([
        fname for fname in os.listdir(gt_dir)
        if fname.startswith("similarityMap_")
    ])

    gt_map = {}

    for fname in tqdm(gt_files, desc="Loading GT files"):
        path = os.path.join(gt_dir, fname)

        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue

                parts = [p.strip() for p in line.split(",") if p.strip()]
                query_idx = int(parts[0])
                neighbor_indices = [int(x) for x in parts[1:]]

                gt_map[query_idx] = neighbor_indices

    return gt_map


gt_map = load_similarity_map(GT_DIR)

print("\n=== GT Load Summary ===")
print(f"Number of GT queries loaded: {len(gt_map)}")
print(f"Min GT query idx          : {min(gt_map.keys()) if gt_map else 'N/A'}")
print(f"Max GT query idx          : {max(gt_map.keys()) if gt_map else 'N/A'}")

Loading GT files: 100%|██████████| 120/120 [03:48<00:00,  1.91s/it]



=== GT Load Summary ===
Number of GT queries loaded: 46754
Min GT query idx          : 187019
Max GT query idx          : 233772


## GT Step 3 — Build DB / Query DataLoaders aligned to real GT

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Prepare batched loaders for:
- database graphs
- query graphs

These loaders are aligned to the real ground-truth indexing scheme.

### Notes
This replaces the earlier pseudo/random test embedding pipeline.

In [36]:
# ============================================================
# GT Step 3 — Build DB / Query DataLoaders aligned to real GT
# Type: Pipeline Code
# Keep later? Yes
# ============================================================

from torch.utils.data import DataLoader
from torch_geometric.data import Batch


MAX_NODES = 500

# Optional first-pass filtering to match training regime
# We trained on graphs <= MAX_NODES, so for a first aligned evaluation,
# we filter both DB and query graphs the same way.
db_graphs_gt_filtered = [g for g in db_graphs_gt if g.num_nodes <= MAX_NODES]
query_graphs_gt_filtered = [g for g in query_graphs_gt if g.num_nodes <= MAX_NODES]

def graph_collate_fn(batch):
    return Batch.from_data_list(batch)

EMBED_BATCH_SIZE = 512

db_loader_gt = DataLoader(
    db_graphs_gt_filtered,
    batch_size=EMBED_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=graph_collate_fn,
    drop_last=False,
)

query_loader_gt = DataLoader(
    query_graphs_gt_filtered,
    batch_size=EMBED_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=graph_collate_fn,
    drop_last=False,
)

print("=== GT-Aligned Embedding Loader Summary ===")
print(f"Original DB graphs          : {len(db_graphs_gt)}")
print(f"Filtered DB graphs          : {len(db_graphs_gt_filtered)}")
print(f"Original Query graphs       : {len(query_graphs_gt)}")
print(f"Filtered Query graphs       : {len(query_graphs_gt_filtered)}")
print(f"DB embedding batches        : {len(db_loader_gt)}")
print(f"Query embedding batches     : {len(query_loader_gt)}")

=== GT-Aligned Embedding Loader Summary ===
Original DB graphs          : 187019
Filtered DB graphs          : 186884
Original Query graphs       : 46532
Filtered Query graphs       : 46507
DB embedding batches        : 366
Query embedding batches     : 91


## GT Step 3 — Generate embeddings for DB and Query sets

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Generate embeddings for:
- database graphs
- query graphs

### Output
For both DB and Query:
- embeddings
- global indices
- poly_ids
- node counts

### Notes
Global indices are critical because GT uses dataset row indices.

In [37]:
# ============================================================
# GT Step 3 — Generate embeddings for DB and Query sets
# Type: Pipeline Code
# Keep later? Yes
# ============================================================

import torch
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GraphConv, global_mean_pool

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


class PolygonGNNEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, embedding_dim):
        super().__init__()

        self.conv1 = GraphConv(in_channels, hidden_channels)
        self.conv2 = GraphConv(hidden_channels, hidden_channels)

        self.lin = nn.Linear(hidden_channels, embedding_dim)

    def forward(self, x, edge_index, batch):
        """
        x: [num_nodes, in_channels]
        edge_index: [2, num_edges]
        batch: [num_nodes] (graph assignment)
        """

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        # pool node features → graph embedding
        x = global_mean_pool(x, batch)

        x = self.lin(x)

        # normalize embeddings
        x = F.normalize(x, p=2, dim=1)

        return x

model = PolygonGNNEncoder(
    in_channels=10,      # IMPORTANT: updated for M2 features
    hidden_channels=64,
    embedding_dim=128
).to(device)

print("Baseline model initialized (untrained).")

def generate_graph_embeddings_with_global_idx(model, loader, device):
    model.eval()

    all_embeddings = []
    all_global_idx = []
    all_poly_ids = []
    all_num_nodes = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Generating embeddings"):
            batch = batch.to(device)

            emb = model(batch.x, batch.edge_index, batch.batch)   # [num_graphs, 128]
            emb = emb.detach().cpu()

            all_embeddings.append(emb)

            # metadata stored per graph
            all_global_idx.extend([int(x) for x in batch.global_idx])
            all_poly_ids.extend(batch.poly_id)

            ptr = batch.ptr.detach().cpu()
            num_nodes_per_graph = (ptr[1:] - ptr[:-1]).tolist()
            all_num_nodes.extend(num_nodes_per_graph)

    all_embeddings = torch.cat(all_embeddings, dim=0)

    return all_embeddings, all_global_idx, all_poly_ids, all_num_nodes


db_embeddings, db_global_idx, db_poly_ids, db_num_nodes = generate_graph_embeddings_with_global_idx(
    model=model,
    loader=db_loader_gt,
    device=device
)

query_embeddings, query_global_idx, query_poly_ids, query_num_nodes = generate_graph_embeddings_with_global_idx(
    model=model,
    loader=query_loader_gt,
    device=device
)

print("\n=== GT Embedding Summary ===")
print("DB embeddings shape    :", tuple(db_embeddings.shape))
print("Num DB global_idx      :", len(db_global_idx))
print("Query embeddings shape :", tuple(query_embeddings.shape))
print("Num Query global_idx   :", len(query_global_idx))

Using device: cuda
Baseline model initialized (untrained).


Generating embeddings: 100%|██████████| 91/91 [00:05<00:00, 17.45it/s]



=== GT Embedding Summary ===
DB embeddings shape    : (186884, 128)
Num DB global_idx      : 186884
Query embeddings shape : (46507, 128)
Num Query global_idx   : 46507


## GT Step 4 — Build DB-only retrieval structures

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Prepare lookup structures for real GT-based retrieval.

### Notes
- Retrieval searches only the DB embedding matrix
- GT uses global dataset indices
- We need DB global-index to row-index mapping for evaluation

In [38]:
# ============================================================
# GT Step 4 — Build DB-only retrieval structures
# Type: Pipeline Code
# Keep later? Yes
# ============================================================

db_global_to_row = {gidx: row_idx for row_idx, gidx in enumerate(db_global_idx)}
query_global_to_row = {gidx: row_idx for row_idx, gidx in enumerate(query_global_idx)}

print("=== Retrieval Structure Summary ===")
print(f"DB rows               : {len(db_global_to_row)}")
print(f"Query rows            : {len(query_global_to_row)}")
print(f"Unique DB global_idx  : {len(set(db_global_idx))}")
print(f"Unique Query global_idx: {len(set(query_global_idx))}")

=== Retrieval Structure Summary ===
DB rows               : 186884
Query rows            : 46507
Unique DB global_idx  : 186884
Unique Query global_idx: 46507


## GT Step 4 — Compute query-to-DB cosine similarity matrix

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Compute cosine similarity between each query embedding and all DB embeddings.

### Notes
Embeddings are already L2-normalized, so cosine similarity is dot product.

In [39]:
# ============================================================
# GT Step 4 — Compute query-to-DB cosine similarity matrix
# Type: Pipeline Code
# Keep later? Yes
# ============================================================

query_db_similarity = query_embeddings @ db_embeddings.T   # [num_queries, num_db]

print("=== Query-DB Similarity Summary ===")
print("shape:", tuple(query_db_similarity.shape))
print("dtype:", query_db_similarity.dtype)

=== Query-DB Similarity Summary ===
shape: (46507, 186884)
dtype: torch.float32


## GT Step 4 — Real Recall@K against similarity map

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Compute true Recall@10 / Recall@50 using:
- predictions from query-to-DB embedding retrieval
- ground truth from `gt_map`

### Evaluation rule
For each query:
- predicted neighbors = top-K DB rows by cosine similarity
- GT neighbors = neighbors from similarity map that are present in filtered DB
- Recall@K = overlap / K

### Query validity
A query is included in Recall@K only if it has at least K valid GT neighbors after DB filtering.

In [40]:
# ============================================================
# GT Step 4 — Real Recall@K against similarity map
# Type: Pipeline Code
# Keep later? Yes
# ============================================================

from tqdm import tqdm

def compute_true_recall_at_k(query_db_similarity, query_global_idx, db_global_idx, db_global_to_row, gt_map, ks=(10, 50)):
    results = {}

    for k in ks:
        recall_sum = 0.0
        valid_queries = 0
        skipped_queries = 0

        for q_row, q_global in tqdm(
            list(enumerate(query_global_idx)),
            desc=f"Computing true Recall@{k}"
        ):
            # GT neighbors for this query
            gt_neighbors_full = gt_map.get(q_global, [])

            # Keep only GT neighbors that exist in the filtered DB embedding set
            gt_neighbors_valid = [gidx for gidx in gt_neighbors_full if gidx in db_global_to_row]

            # Need at least K valid GT neighbors to evaluate Recall@K fairly
            if len(gt_neighbors_valid) < k:
                skipped_queries += 1
                continue

            gt_topk = gt_neighbors_valid[:k]
            gt_topk_set = set(gt_topk)

            sims = query_db_similarity[q_row]
            topk_scores, topk_db_rows = torch.topk(sims, k=k, largest=True)

            pred_global = [db_global_idx[row] for row in topk_db_rows.tolist()]
            pred_set = set(pred_global)

            overlap = len(pred_set & gt_topk_set)
            recall = overlap / k

            recall_sum += recall
            valid_queries += 1

        results[k] = {
            "recall": recall_sum / max(valid_queries, 1),
            "valid_queries": valid_queries,
            "skipped_queries": skipped_queries,
        }

    return results

## GT Step 4 Test — Run true Recall@10 / Recall@50

**Type:** Evaluation Code  
**Keep later?** Yes

### Purpose
Measure true retrieval performance against the real similarity map.

### Output
- Recall@10
- Recall@50
- number of valid and skipped queries

In [41]:
# ============================================================
# GT Step 4 Test — Run true Recall@10 / Recall@50
# Type: Evaluation Code
# Keep later? Yes
# ============================================================

true_recall_results = compute_true_recall_at_k(
    query_db_similarity=query_db_similarity,
    query_global_idx=query_global_idx,
    db_global_idx=db_global_idx,
    db_global_to_row=db_global_to_row,
    gt_map=gt_map,
    ks=(10, 50),
)

print("\n=== True Recall@K Results ===")
for k, stats in true_recall_results.items():
    print(
        f"Recall@{k}: {stats['recall']:.4f} | "
        f"valid_queries={stats['valid_queries']} | "
        f"skipped_queries={stats['skipped_queries']}"
    )

Computing true Recall@50: 100%|██████████| 46507/46507 [02:42<00:00, 286.42it/s]



=== True Recall@K Results ===
Recall@10: 0.0001 | valid_queries=42891 | skipped_queries=3616
Recall@50: 0.0003 | valid_queries=40897 | skipped_queries=5610


## GT Step 5A — GT-specific collate function

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Batch GT-supervised triplets correctly.

### Why needed
The GT dataset will return:
- `anchor`
- `positive`
- `negative`
- `anchor_global_idx`
- `positive_global_idx`
- `negative_global_idx`

So we need a GT-specific collate function for the DataLoader.

In [42]:
# ============================================================
# GT Step 5A — GT-specific collate function
# Type: Pipeline Code
# Keep later? Yes
# ============================================================

import torch
from torch_geometric.data import Batch

def gt_triplet_collate_fn(batch):
    """
    Collate GT-supervised triplet samples into 3 PyG batches.
    """
    anchor_list = [item["anchor"] for item in batch]
    positive_list = [item["positive"] for item in batch]
    negative_list = [item["negative"] for item in batch]

    anchor_batch = Batch.from_data_list(anchor_list)
    positive_batch = Batch.from_data_list(positive_list)
    negative_batch = Batch.from_data_list(negative_list)

    anchor_global_idx = torch.tensor(
        [item["anchor_global_idx"] for item in batch],
        dtype=torch.long
    )
    positive_global_idx = torch.tensor(
        [item["positive_global_idx"] for item in batch],
        dtype=torch.long
    )
    negative_global_idx = torch.tensor(
        [item["negative_global_idx"] for item in batch],
        dtype=torch.long
    )

    return {
        "anchor_batch": anchor_batch,
        "positive_batch": positive_batch,
        "negative_batch": negative_batch,
        "anchor_global_idx": anchor_global_idx,
        "positive_global_idx": positive_global_idx,
        "negative_global_idx": negative_global_idx,
    }

## GT Step 5B — Build DB node-count buckets for hard negative sampling

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Group filtered DB graphs by node-count buckets so negatives can be sampled from DB graphs of similar size.

### Why
Random negatives are too easy.
Same-size-bucket negatives are harder and should provide a stronger training signal.

### Output
- `db_bucket_to_global_idx`
- `db_global_to_bucket`

In [43]:
# ============================================================
# GT Step 5B — Build DB node-count buckets for hard negative sampling
# Type: Pipeline Code
# Keep later? Yes
# ============================================================

from collections import defaultdict
from tqdm import tqdm

GT_DB_NODE_BUCKET_SIZE = 5

db_bucket_to_global_idx = defaultdict(list)
db_global_to_bucket = {}

for g in tqdm(db_graphs_gt_filtered, desc="Building DB size buckets"):
    bucket_id = g.num_nodes // GT_DB_NODE_BUCKET_SIZE
    db_global_to_bucket[g.global_idx] = bucket_id
    db_bucket_to_global_idx[bucket_id].append(g.global_idx)

print("=== GT DB Bucket Summary ===")
print(f"Filtered DB graphs : {len(db_graphs_gt_filtered)}")
print(f"Number of buckets  : {len(db_bucket_to_global_idx)}")

Building DB size buckets: 100%|██████████| 186884/186884 [00:01<00:00, 114015.34it/s]

=== GT DB Bucket Summary ===
Filtered DB graphs : 186884
Number of buckets  : 99


In [44]:
# ============================================================
# GT Step 5B Test — Inspect DB bucket distribution
# Type: Diagnostic / Inspection Code
# Keep later? No
# ============================================================

bucket_sizes = {
    bucket_id: len(indices)
    for bucket_id, indices in db_bucket_to_global_idx.items()
}

print("Top 10 largest DB buckets:")
for bucket_id, size in sorted(bucket_sizes.items(), key=lambda x: x[1], reverse=True)[:10]:
    low = bucket_id * GT_DB_NODE_BUCKET_SIZE
    high = low + GT_DB_NODE_BUCKET_SIZE - 1
    print(f"Bucket {bucket_id:3d} ({low:3d}-{high:3d} nodes): {size}")

print("\nTop 10 smallest DB buckets:")
for bucket_id, size in sorted(bucket_sizes.items(), key=lambda x: x[1])[:10]:
    low = bucket_id * GT_DB_NODE_BUCKET_SIZE
    high = low + GT_DB_NODE_BUCKET_SIZE - 1
    print(f"Bucket {bucket_id:3d} ({low:3d}-{high:3d} nodes): {size}")

Top 10 largest DB buckets:
Bucket   1 (  5-  9 nodes): 57814
Bucket   0 (  0-  4 nodes): 42945
Bucket   2 ( 10- 14 nodes): 31015
Bucket   3 ( 15- 19 nodes): 18381
Bucket   4 ( 20- 24 nodes): 10767
Bucket   5 ( 25- 29 nodes): 6723
Bucket   6 ( 30- 34 nodes): 4564
Bucket   7 ( 35- 39 nodes): 3104
Bucket   8 ( 40- 44 nodes): 2315
Bucket   9 ( 45- 49 nodes): 1670

Top 10 smallest DB buckets:
Bucket  87 (435-439 nodes): 1
Bucket  85 (425-429 nodes): 1
Bucket  93 (465-469 nodes): 2
Bucket  90 (450-454 nodes): 2
Bucket  89 (445-449 nodes): 2
Bucket  99 (495-499 nodes): 2
Bucket  97 (485-489 nodes): 2
Bucket  86 (430-434 nodes): 2
Bucket  91 (455-459 nodes): 2
Bucket  83 (415-419 nodes): 2


## GT Step 5C — Rebuild GT positive pools using only top-K positives

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Restrict positive sampling to the top-K GT neighbors only.

### Why
Some GT pools are extremely large, which makes random positive sampling too noisy.
Using only the top-ranked GT neighbors gives much cleaner supervision.

### Current choice
- `TOP_POS_K = 10`

### Output
- `db_graph_by_global_idx`
- `query_graph_by_global_idx`
- `gt_train_samples_topk`

In [45]:
# ============================================================
# GT Step 5C — Rebuild GT positive pools using only top-K positives
# Type: Pipeline Code
# Keep later? Yes
# ============================================================

TOP_POS_K = 10

# Fast lookup: filtered DB global_idx -> graph object
db_graph_by_global_idx = {g.global_idx: g for g in db_graphs_gt_filtered}

# Fast lookup: filtered Query global_idx -> graph object
query_graph_by_global_idx = {g.global_idx: g for g in query_graphs_gt_filtered}

gt_train_samples_topk = []
queries_without_valid_topk_positive = 0

for q_global in tqdm(query_global_idx, desc="Building top-K GT positive pools"):
    gt_neighbors_full = gt_map.get(q_global, [])

    # keep GT ranking order, but only top-K valid filtered DB neighbors
    valid_pos_global = []
    for gidx in gt_neighbors_full:
        if gidx in db_graph_by_global_idx:
            valid_pos_global.append(gidx)
        if len(valid_pos_global) >= TOP_POS_K:
            break

    if len(valid_pos_global) == 0:
        queries_without_valid_topk_positive += 1
        continue

    gt_train_samples_topk.append({
        "query_global_idx": q_global,
        "positive_global_pool": valid_pos_global,
    })

print("\n=== Top-K GT Positive Pool Summary ===")
print(f"Filtered query graphs                 : {len(query_graphs_gt_filtered)}")
print(f"Queries with >=1 valid top-K positive : {len(gt_train_samples_topk)}")
print(f"Queries with 0 valid top-K positive   : {queries_without_valid_topk_positive}")
print(f"TOP_POS_K                             : {TOP_POS_K}")

Building top-K GT positive pools: 100%|██████████| 46507/46507 [00:00<00:00, 120249.63it/s]


=== Top-K GT Positive Pool Summary ===
Filtered query graphs                 : 46507
Queries with >=1 valid top-K positive : 44429
Queries with 0 valid top-K positive   : 2078
TOP_POS_K                             : 10


## GT Step 5D — Build GT triplet dataset with top-K positives and hard negatives

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Create a stronger GT-supervised triplet dataset.

### Training design
- anchor = query graph
- positive = sampled from top-K GT neighbors
- negative = sampled from same-size DB bucket, excluding GT neighbors

### Fallback
If a same-bucket non-GT negative is unavailable, expand to nearby buckets.
If still unavailable, fall back to random non-GT DB negative.

In [49]:
# ============================================================
# GT Step 5D — Build GT triplet dataset with top-K positives and hard negatives
# Type: Pipeline Code
# Keep later? Yes
# ============================================================

import random
from torch.utils.data import Dataset


class RealGTTripletDatasetTopKHardNeg(Dataset):
    def __init__(
        self,
        gt_train_samples,
        query_graph_by_global_idx,
        db_graph_by_global_idx,
        db_global_idx,
        db_global_to_bucket,
        db_bucket_to_global_idx,
        node_bucket_size=5,
        max_bucket_hops=2,
        seed=42
    ):
        self.gt_train_samples = gt_train_samples
        self.query_graph_by_global_idx = query_graph_by_global_idx
        self.db_graph_by_global_idx = db_graph_by_global_idx
        self.db_global_idx = db_global_idx
        self.db_global_to_bucket = db_global_to_bucket
        self.db_bucket_to_global_idx = db_bucket_to_global_idx
        self.node_bucket_size = node_bucket_size
        self.max_bucket_hops = max_bucket_hops
        self.rng = random.Random(seed)

    def __len__(self):
        return len(self.gt_train_samples)

    def _sample_random_non_gt_negative(self, positive_set):
        neg_global = self.rng.choice(self.db_global_idx)
        while neg_global in positive_set:
            neg_global = self.rng.choice(self.db_global_idx)
        return neg_global, "random_non_gt"

    def _sample_bucket_hard_negative(self, anchor_num_nodes, positive_set):
        anchor_bucket = anchor_num_nodes // self.node_bucket_size

        # same bucket first
        same_bucket = [
            gidx for gidx in self.db_bucket_to_global_idx.get(anchor_bucket, [])
            if gidx not in positive_set
        ]
        if same_bucket:
            return self.rng.choice(same_bucket), "same_bucket_non_gt"

        # expand to neighboring buckets
        for hop in range(1, self.max_bucket_hops + 1):
            expanded = []

            left_bucket = anchor_bucket - hop
            right_bucket = anchor_bucket + hop

            if left_bucket in self.db_bucket_to_global_idx:
                expanded.extend([
                    gidx for gidx in self.db_bucket_to_global_idx[left_bucket]
                    if gidx not in positive_set
                ])

            if right_bucket in self.db_bucket_to_global_idx:
                expanded.extend([
                    gidx for gidx in self.db_bucket_to_global_idx[right_bucket]
                    if gidx not in positive_set
                ])

            if expanded:
                return self.rng.choice(expanded), f"neighbor_bucket_non_gt_hop_{hop}"

        # fallback
        return self._sample_random_non_gt_negative(positive_set)

    def __getitem__(self, idx):
        item = self.gt_train_samples[idx]

        q_global = item["query_global_idx"]
        pos_pool = item["positive_global_pool"]

        anchor = self.query_graph_by_global_idx[q_global]
        pos_global = self.rng.choice(pos_pool)
        positive = self.db_graph_by_global_idx[pos_global]

        positive_set = set(pos_pool)
        neg_global, neg_source = self._sample_bucket_hard_negative(anchor.num_nodes, positive_set)
        negative = self.db_graph_by_global_idx[neg_global]

        return {
            "anchor": anchor,
            "positive": positive,
            "negative": negative,
            "anchor_global_idx": q_global,
            "positive_global_idx": pos_global,
            "negative_global_idx": neg_global,
            "negative_source": neg_source,
        }

## GT Step 5D — Instantiate GT triplet dataset

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Create the actual GT triplet dataset object using:
- top-K GT positives
- same-size-bucket hard negatives

### Output
- `gt_triplet_dataset_topk_hard`

In [51]:
# ============================================================
# GT Step 5D — Instantiate GT triplet dataset
# Type: Pipeline Code
# Keep later? Yes
# ============================================================

gt_triplet_dataset_topk_hard = RealGTTripletDatasetTopKHardNeg(
    gt_train_samples=gt_train_samples_topk,
    query_graph_by_global_idx=query_graph_by_global_idx,
    db_graph_by_global_idx=db_graph_by_global_idx,
    db_global_idx=[g.global_idx for g in db_graphs_gt_filtered],
    db_global_to_bucket=db_global_to_bucket,
    db_bucket_to_global_idx=db_bucket_to_global_idx,
    node_bucket_size=GT_DB_NODE_BUCKET_SIZE,
    max_bucket_hops=2,
    seed=42
)

print("GT triplet dataset created.")
print("Dataset length:", len(gt_triplet_dataset_topk_hard))

GT triplet dataset created.
Dataset length: 44429


In [50]:
print("graph_data_list" in globals())
print("normalized_graph_data_list" in globals())
print("db_graphs_gt" in globals())
print("query_graphs_gt" in globals())
print("gt_map" in globals())
print("db_graphs_gt_filtered" in globals())
print("query_graphs_gt_filtered" in globals())
print("db_bucket_to_global_idx" in globals())
print("db_global_to_bucket" in globals())
print("gt_train_samples_topk" in globals())
print("gt_triplet_dataset_topk_hard" in globals())

True
True
True
True
True
True
True
True
True
True
False


## GT Step 5E — Build GT training DataLoader

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Create the training DataLoader for GT-supervised triplet learning using:
- top-K GT positives
- same-size-bucket hard negatives

### Output
- `gt_train_loader_topk_hard`

### Notes
This loader will be used for the clean GT-aligned training run.

In [52]:
# ============================================================
# GT Step 5E — Build GT training DataLoader
# Type: Pipeline Code
# Keep later? Yes
# ============================================================

from torch.utils.data import DataLoader

GT_TOPK_HARD_BATCH_SIZE = 256

gt_train_loader_topk_hard = DataLoader(
    gt_triplet_dataset_topk_hard,
    batch_size=GT_TOPK_HARD_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    collate_fn=gt_triplet_collate_fn,
    drop_last=True,
)

print("=== GT Train Loader Summary ===")
print(f"Dataset size  : {len(gt_triplet_dataset_topk_hard)}")
print(f"Train batches : {len(gt_train_loader_topk_hard)}")

=== GT Train Loader Summary ===
Dataset size  : 44429
Train batches : 173


## GT Step 6A — Train one full GT-supervised epoch

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Train the PolygonGNN model for one full epoch using real GT-supervised triplets.

### Training signal
- anchor = query graph
- positive = top-K GT neighbor from DB
- negative = same-size-bucket non-GT DB graph

### Output
- average GT training loss for the epoch

In [53]:
# ============================================================
# GT Step 6A — Train one full GT-supervised epoch
# Type: Pipeline Code
# Keep later? Yes
# ============================================================

from tqdm import tqdm
import torch

def train_one_epoch_gt(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    total_batches = 0

    progress_bar = tqdm(loader, desc="GT training epoch", leave=True)

    for batch in progress_bar:
        a_batch = batch["anchor_batch"].to(device)
        p_batch = batch["positive_batch"].to(device)
        n_batch = batch["negative_batch"].to(device)

        optimizer.zero_grad()

        a_emb = model(a_batch.x, a_batch.edge_index, a_batch.batch)
        p_emb = model(p_batch.x, p_batch.edge_index, p_batch.batch)
        n_emb = model(n_batch.x, n_batch.edge_index, n_batch.batch)

        loss = criterion(a_emb, p_emb, n_emb)

        if not torch.isfinite(loss):
            raise ValueError(f"Non-finite GT loss encountered: {loss.item()}")

        loss.backward()
        optimizer.step()

        loss_value = loss.item()
        total_loss += loss_value
        total_batches += 1

        avg_loss = total_loss / total_batches
        progress_bar.set_postfix({
            "batch_loss": f"{loss_value:.4f}",
            "avg_loss": f"{avg_loss:.4f}"
        })

    epoch_loss = total_loss / max(total_batches, 1)
    return epoch_loss

## GT Step 6B — Reinitialize fresh GT-supervised model for 10D features

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Create a fresh PolygonGNN model that matches the upgraded 10D node features.

### Important
This must use:
- `in_channels=10`

Otherwise training will fail with a shape mismatch.

In [56]:
# ============================================================
# GT Step 6B — Reinitialize fresh GT-supervised model for 10D features
# Type: Pipeline Code
# Keep later? Yes
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

gt_model_topk_hard = PolygonGNNEncoder(
    in_channels=10,   # IMPORTANT: upgraded feature size
    hidden_channels=64,
    embedding_dim=128
).to(device)

gt_criterion_topk_hard = nn.TripletMarginLoss(margin=0.2, p=2)
gt_optimizer_topk_hard = optim.Adam(
    gt_model_topk_hard.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

print("Fresh GT-supervised 10D model initialized.")

Using device: cuda
Fresh GT-supervised 10D model initialized.


## GT Step 6C — Train for 3 epochs

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Train the GT-supervised PolygonGNN model for a small but meaningful number of epochs.

### Training setup
- anchor = query graph
- positive = top-K GT neighbor from DB
- negative = same-size-bucket non-GT DB graph

### Output
- per-epoch GT training loss
- trained model ready for fresh embedding generation and real GT evaluation

In [58]:
# ============================================================
# GT Step 6C — Train for 3 epochs
# Type: Pipeline Code
# Keep later? Yes
# ============================================================

NUM_GT_EPOCHS = 3
gt_topk_hard_train_losses = []

for epoch in range(1, NUM_GT_EPOCHS + 1):
    print(f"\n===== GT Training | Epoch {epoch}/{NUM_GT_EPOCHS} =====")

    epoch_loss = train_one_epoch_gt(
        model=gt_model_topk_hard,
        loader=gt_train_loader_topk_hard,
        optimizer=gt_optimizer_topk_hard,
        criterion=gt_criterion_topk_hard,
        device=device
    )

    gt_topk_hard_train_losses.append(epoch_loss)
    print(f"Epoch {epoch} train loss: {epoch_loss:.6f}")


===== GT Training | Epoch 1/3 =====


GT training epoch: 100%|██████████| 173/173 [01:32<00:00,  1.87it/s, batch_loss=0.1986, avg_loss=0.2050]


Epoch 1 train loss: 0.205002

===== GT Training | Epoch 2/3 =====


GT training epoch: 100%|██████████| 173/173 [01:30<00:00,  1.92it/s, batch_loss=0.2034, avg_loss=0.2006]


Epoch 2 train loss: 0.200557

===== GT Training | Epoch 3/3 =====


GT training epoch: 100%|██████████| 173/173 [01:26<00:00,  1.99it/s, batch_loss=0.1992, avg_loss=0.2002]

Epoch 3 train loss: 0.200208


## GT Step 7A — Generate embeddings using trained 10D model

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Generate fresh embeddings after training with richer node features.

### Output
- DB embeddings
- Query embeddings
- aligned global indices

In [59]:
# ============================================================
# GT Step 7A — Generate embeddings using trained 10D model
# ============================================================

gt_db_embeddings, gt_db_global_idx, _, _ = generate_graph_embeddings_with_global_idx(
    model=gt_model_topk_hard,
    loader=db_loader_gt,
    device=device
)

gt_query_embeddings, gt_query_global_idx, _, _ = generate_graph_embeddings_with_global_idx(
    model=gt_model_topk_hard,
    loader=query_loader_gt,
    device=device
)

print("\n=== Embedding Summary ===")
print("DB embeddings shape    :", gt_db_embeddings.shape)
print("Query embeddings shape :", gt_query_embeddings.shape)

Generating embeddings: 100%|██████████| 91/91 [00:03<00:00, 26.93it/s]


=== Embedding Summary ===
DB embeddings shape    : torch.Size([186884, 128])
Query embeddings shape : torch.Size([46507, 128])


In [61]:
# ============================================================
# GT Step 7B — Build retrieval structures
# ============================================================

gt_db_global_to_row = {
    gidx: i for i, gidx in enumerate(gt_db_global_idx)
}

gt_query_db_similarity = gt_query_embeddings @ gt_db_embeddings.T

print("Similarity shape:", gt_query_db_similarity.shape)

Similarity shape: torch.Size([46507, 186884])


## Utility — Fast chunked Recall@K computation

**Type:** Pipeline Code  
**Keep later?** Yes

### Purpose
Efficiently compute Recall@K using chunked top-k evaluation.

### Why
Avoids slow per-query loops and avoids multiprocessing overhead.

In [63]:
# ============================================================
# Utility — Fast chunked Recall@K computation
# ============================================================

import torch
from tqdm import tqdm

def compute_true_recall_at_k_chunked(
    query_db_similarity,
    query_global_idx,
    db_global_idx,
    db_global_to_row,
    gt_map,
    ks=(10, 50),
    chunk_size=1024
):
    max_k = max(ks)
    num_queries = len(query_global_idx)

    recall_sums = {k: 0.0 for k in ks}
    valid_counts = {k: 0 for k in ks}
    skipped_counts = {k: 0 for k in ks}

    for start in tqdm(range(0, num_queries, chunk_size), desc="Chunked Recall@K"):
        end = min(start + chunk_size, num_queries)

        sim_chunk = query_db_similarity[start:end]
        _, topk_rows_chunk = torch.topk(sim_chunk, k=max_k, dim=1)

        topk_rows_chunk = topk_rows_chunk.cpu()

        for i in range(end - start):
            q_row = start + i
            q_global = query_global_idx[q_row]

            gt_neighbors = gt_map.get(q_global, [])
            gt_neighbors_valid = [
                g for g in gt_neighbors if g in db_global_to_row
            ]

            pred_rows = topk_rows_chunk[i].tolist()
            pred_global = [db_global_idx[r] for r in pred_rows]

            for k in ks:
                if len(gt_neighbors_valid) < k:
                    skipped_counts[k] += 1
                    continue

                gt_set = set(gt_neighbors_valid[:k])
                pred_set = set(pred_global[:k])

                overlap = len(gt_set & pred_set)

                recall_sums[k] += overlap / k
                valid_counts[k] += 1

    results = {}
    for k in ks:
        results[k] = {
            "recall": recall_sums[k] / max(valid_counts[k], 1),
            "valid_queries": valid_counts[k],
            "skipped_queries": skipped_counts[k],
        }

    return results

In [64]:
# ============================================================
# GT Step 7C — Compute true Recall@K (fast)
# ============================================================

results = compute_true_recall_at_k_chunked(
    query_db_similarity=gt_query_db_similarity,
    query_global_idx=gt_query_global_idx,
    db_global_idx=gt_db_global_idx,
    db_global_to_row=gt_db_global_to_row,
    gt_map=gt_map,
    ks=(10, 50),
    chunk_size=1024
)

print("\n=== NEW True Recall@K ===")
for k, stats in results.items():
    print(
        f"Recall@{k}: {stats['recall']:.4f} | "
        f"valid={stats['valid_queries']} | "
        f"skipped={stats['skipped_queries']}"
    )

Chunked Recall@K: 100%|██████████| 46/46 [00:30<00:00,  1.49it/s]


=== NEW True Recall@K ===
Recall@10: 0.0001 | valid=42891 | skipped=3616
Recall@50: 0.0003 | valid=40897 | skipped=5610
